In [ ]:
%pip -q install "stable-baselines3[extra]" gymnasium pytorch-fid seaborn scikit-learn tqdm

print("\n✅ Dependency installation completed: required Python packages are available for the experiment.")


# Medical Image Generation for Rare Diseases

RL-guided cGAN latent-space exploration for HAM10000 rare skin lesion classes.

### Phase 0.1: Imports and Scientific Stack

This cell loads the numerical, deep learning, visualization, RL, and evaluation libraries used throughout the project.

In [ ]:
import os
import sys
import math
import json
import time
import copy
import random
import shutil
import warnings
import subprocess
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as T
from torchvision import models
from torchvision.utils import make_grid, save_image

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE

try:
    from sklearn.model_selection import StratifiedGroupKFold
except Exception:
    StratifiedGroupKFold = None

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

print("\n✅ Phase 0.1 completed: Imports and Scientific Stack executed successfully.")

### Phase 0.2: Runtime Configuration

Global constants define the target classes, latent dimensionality, image sizes, device placement, and training profiles.

In [ ]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
RUN_MODE = "full"
USE_CACHED = os.environ.get("USE_CACHED", "false").lower() == "true"
EXPERIMENT_VERSION = "rare_aware_v5_fid_balanced"

CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}
RARE_CLASSES = ["akiec", "df", "vasc"]
RARE_IDS = [CLASS_TO_IDX[name] for name in RARE_CLASSES]

LATENT_DIM = 128
N_CLASSES = len(CLASS_NAMES)
GAN_IMAGE_SIZE = 64
CLS_IMAGE_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints" / EXPERIMENT_VERSION
GENERATED_DIR = OUTPUT_DIR / "generated" / EXPERIMENT_VERSION
PLOT_DIR = OUTPUT_DIR / "plots" / EXPERIMENT_VERSION
for folder in [CHECKPOINT_DIR, GENERATED_DIR, PLOT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PROFILES = {
    "debug": {
        "gan_epochs": 12,
        "classifier_epochs": 10,
        "ppo_timesteps": 30_000,
        "eval_per_class": 60,
        "ppo_eval_restarts": 4,
        "ppo_candidate_multiplier": 3,
        "gan_batch": 128,
        "classifier_batch": 48,
        "classifier_sampler_multiplier": 1.8,
        "gan_lr_g": 1.0e-4,
        "gan_lr_d": 2.0e-4,
        "gan_ema_decay": 0.995,
        "r1_gamma": 5.0,
        "r1_interval": 8,
        "gan_diff_augment": True,
        "run_fid": False,
    },
    "full": {
        "gan_epochs": 140,
        "classifier_epochs": 45,
        "ppo_timesteps": 300_000,
        "eval_per_class": 300,
        "ppo_eval_restarts": 10,
        "ppo_candidate_multiplier": 4,
        "gan_batch": 128,
        "classifier_batch": 48,
        "classifier_sampler_multiplier": 2.0,
        "gan_lr_g": 1.0e-4,
        "gan_lr_d": 2.0e-4,
        "gan_ema_decay": 0.999,
        "r1_gamma": 5.0,
        "r1_interval": 8,
        "gan_diff_augment": True,
        "run_fid": True,
    },
}
CFG = PROFILES["full"] if RUN_MODE == "full" else PROFILES["debug"]

print("\n✅ Phase 0.2 completed: Runtime Configuration executed successfully.")

### Phase 0.3: Reproducibility Utilities

The experiment is seeded and the active runtime profile is printed for transparent reporting.

In [ ]:
def seed_everything(seed=42):
    """Seed Python, NumPy, PyTorch, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def show_config():
    """Display the active runtime profile."""
    print(f"device={DEVICE}")
    print(f"run_mode={RUN_MODE}")
    print(json.dumps(CFG, indent=2))


seed_everything(SEED)
print("\n⚙️ Runtime configuration: device, execution mode, and training profile are reported for reproducible execution.")
show_config()

print("\n✅ Phase 0.3 completed: Reproducibility Utilities executed successfully.")


### Phase 1: Data Preparation

HAM10000 is prepared for both generative modeling and classifier-based medical evaluation.

### Phase 1.1: Dataset Discovery

The notebook locates HAM10000 inside Kaggle input storage, with a credential-based fallback for notebook environments.

In [ ]:
DATASET_SLUG = "kmader/skin-cancer-mnist-ham10000"


def find_ham10000_root():
    """Find HAM10000 inside Kaggle input or download it when credentials exist."""
    search_roots = [Path("/kaggle/input"), Path("/content"), Path.cwd()]
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob("HAM10000_metadata.csv"))
            if matches:
                return matches[0].parent

    download_root = OUTPUT_DIR / "ham10000"
    download_root.mkdir(parents=True, exist_ok=True)
    has_kaggle_auth = Path.home().joinpath(".kaggle", "kaggle.json").exists() or os.getenv("KAGGLE_USERNAME")
    if has_kaggle_auth:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", str(download_root), "--unzip"],
            check=True,
        )
        matches = list(download_root.rglob("HAM10000_metadata.csv"))
        if matches:
            return matches[0].parent

    raise FileNotFoundError(
        "Add the Kaggle dataset to the notebook: kmader/skin-cancer-mnist-ham10000"
    )

print("\n✅ Phase 1.1 completed: Dataset Discovery executed successfully.")


### Phase 1.2: Metadata and Image Paths

Metadata are loaded, image paths are attached, and diagnostic labels are mapped to numeric classes.

In [ ]:
def load_ham10000_metadata(data_root):
    """Load metadata and attach image paths."""
    metadata_path = next(Path(data_root).rglob("HAM10000_metadata.csv"))
    df = pd.read_csv(metadata_path)

    image_paths = {}
    for ext in ["jpg", "jpeg", "png", "JPG", "JPEG", "PNG"]:
        for path in Path(data_root).rglob(f"*.{ext}"):
            image_paths[path.stem] = str(path)

    df["path"] = df["image_id"].map(image_paths)
    df = df.dropna(subset=["path", "dx"]).copy()
    df = df[df["dx"].isin(CLASS_NAMES)].reset_index(drop=True)
    df["label"] = df["dx"].map(CLASS_TO_IDX).astype(int)
    df["is_rare"] = df["dx"].isin(RARE_CLASSES)
    return df

print("\n✅ Phase 1.2 completed: Metadata and Image Paths executed successfully.")


### Phase 1.3: Stratified Validation Split

The split preserves class proportions and uses lesion-level grouping when available.

In [ ]:
def split_by_lesion(df, val_fraction=0.2):
    """Create a stratified validation split with lesion-level grouping when available."""
    if StratifiedGroupKFold is not None and "lesion_id" in df.columns:
        n_splits = max(2, round(1 / val_fraction))
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        try:
            train_idx, val_idx = next(splitter.split(df, y=df["dx"], groups=df["lesion_id"]))
            return df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
        except Exception:
            pass

    from sklearn.model_selection import train_test_split

    train_df, val_df = train_test_split(
        df,
        test_size=val_fraction,
        random_state=SEED,
        stratify=df["dx"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print("\n✅ Phase 1.3 completed: Stratified Validation Split executed successfully.")


### Phase 1.4: Class Imbalance Audit

The class distribution is displayed to motivate rare-disease generation and balanced sampling.

In [ ]:
DATA_ROOT = find_ham10000_root()
df = load_ham10000_metadata(DATA_ROOT)
train_df, val_df = split_by_lesion(df)

counts = df["dx"].value_counts().reindex(CLASS_NAMES)
print("\n📊 Class-count table: this dataframe quantifies HAM10000 class imbalance and identifies the rare diagnostic classes targeted by PPO.")
display(pd.DataFrame({"class": counts.index, "count": counts.values, "rare": counts.index.isin(RARE_CLASSES)}))

print("\n📊 Class-distribution plot: this bar chart motivates class-balanced sampling and RL-guided rare-disease generation.")
plt.figure(figsize=(9, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index.isin(RARE_CLASSES), dodge=False, palette=["#4C78A8", "#E45756"])
plt.title("Figure 1: HAM10000 Diagnostic Class Distribution")
plt.xlabel("Diagnostic class")
plt.ylabel("Number of images")
plt.legend(title="Rare class")
plt.tight_layout()
plt.show()

print("\n⚙️ Dataset split summary: data location and train-validation counts are reported for experimental traceability.")
print(f"data_root={DATA_ROOT}")
print(f"train={len(train_df)} validation={len(val_df)}")

print("\n✅ Phase 1.4 completed: Class Imbalance Audit executed successfully.")


### Phase 1.5: Image Transform Pipelines

GAN images are normalized to [-1, 1], while classifier images use ImageNet normalization and stronger rare-class augmentation.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NUM_WORKERS = 0


def optional_elastic():
    """Return elastic augmentation when torchvision supports it."""
    if hasattr(T, "ElasticTransform"):
        return T.ElasticTransform(alpha=35.0, sigma=5.0)
    return T.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.95, 1.05))


gan_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

cls_common_transform = T.Compose([
    T.RandomResizedCrop(CLS_IMAGE_SIZE, scale=(0.82, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cls_rare_transform = T.Compose([
    T.Resize((CLS_IMAGE_SIZE, CLS_IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.RandomAffine(degrees=0, translate=(0.07, 0.07), scale=(0.92, 1.10)),
    T.RandomApply([optional_elastic()], p=0.35),
    T.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.14, hue=0.025),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cls_eval_transform = T.Compose([
    T.Resize((CLS_IMAGE_SIZE, CLS_IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("\n✅ Phase 1.5 completed: Image Transform Pipelines executed successfully.")

### Phase 1.6: Dataset Wrapper and Balanced Sampler

The dataset wrapper serves both GAN and classifier training, and inverse-frequency sampling offsets class imbalance.

In [ ]:
class HAM10000Dataset(Dataset):
    """HAM10000 image dataset for GAN or classifier training."""
    def __init__(self, frame, target="gan", train=True):
        self.frame = frame.reset_index(drop=True)
        self.target = target
        self.train = train

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        label = int(row["label"])

        if self.target == "gan":
            image = gan_transform(image)
        elif self.train and label in RARE_IDS:
            image = cls_rare_transform(image)
        elif self.train:
            image = cls_common_transform(image)
        else:
            image = cls_eval_transform(image)

        return image, label


def weighted_sampler(frame, multiplier=1.0):
    """Build a class-balanced sampler."""
    counts = frame["label"].value_counts().to_dict()
    weights = frame["label"].map(lambda label: 1.0 / counts[int(label)]).to_numpy()
    num_samples = int(len(weights) * multiplier)
    return WeightedRandomSampler(weights, num_samples=num_samples, replacement=True)

print("\n✅ Phase 1.6 completed: Dataset Wrapper and Balanced Sampler executed successfully.")

### Phase 1.7: DataLoaders

DataLoaders are created for cGAN training, classifier training, and validation evaluation.

In [ ]:
gan_train_ds = HAM10000Dataset(train_df, target="gan", train=True)
cls_train_ds = HAM10000Dataset(train_df, target="classifier", train=True)
cls_val_ds = HAM10000Dataset(val_df, target="classifier", train=False)

gan_loader = DataLoader(
    gan_train_ds,
    batch_size=CFG["gan_batch"],
    sampler=weighted_sampler(train_df),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

cls_train_loader = DataLoader(
    cls_train_ds,
    batch_size=CFG["classifier_batch"],
    sampler=weighted_sampler(train_df, multiplier=CFG["classifier_sampler_multiplier"]),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

cls_val_loader = DataLoader(
    cls_val_ds,
    batch_size=CFG["classifier_batch"],
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("\n⚙️ DataLoader batch counts: values correspond to GAN training, classifier training, and classifier validation pipelines.")
print(len(gan_loader), len(cls_train_loader), len(cls_val_loader))

print("\n✅ Phase 1.7 completed: DataLoaders executed successfully.")

### Phase 2: Conditional GAN

A conditional DCGAN learns a label-aware image manifold that the PPO agent later explores.

### Phase 2.1: Shared Model Utilities

One-hot label encoding and DCGAN initialization are defined once for reuse.

In [ ]:
def one_hot(labels, n_classes=N_CLASSES):
    """Convert labels to one-hot vectors."""
    return F.one_hot(labels.long(), n_classes).float()


def weights_init(module):
    """DCGAN initialization."""
    name = module.__class__.__name__
    if "Conv" in name or "Linear" in name:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0)

print("\n✅ Phase 2.1 completed: Shared Model Utilities executed successfully.")


### Phase 2.2: Conditional Generator

The generator maps a latent vector and disease label to a 64x64 RGB lesion image.

In [ ]:
class ConditionalGenerator(nn.Module):
    """Conditional DCGAN generator."""
    def __init__(self, latent_dim=128, n_classes=7, ngf=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.n_classes = n_classes
        self.project = nn.Sequential(
            nn.Linear(latent_dim + n_classes, ngf * 8 * 4 * 4),
            nn.BatchNorm1d(ngf * 8 * 4 * 4),
            nn.ReLU(True),
        )
        self.net = nn.Sequential(
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        y = one_hot(labels, self.n_classes).to(z.device)
        x = torch.cat([z, y], dim=1)
        x = self.project(x).view(z.size(0), -1, 4, 4)
        return self.net(x)

print("\n✅ Phase 2.2 completed: Conditional Generator executed successfully.")


### Phase 2.3: Conditional Discriminator

The discriminator evaluates realism while conditioning on the same disease label.

In [ ]:
class ConditionalDiscriminator(nn.Module):
    """Conditional DCGAN discriminator."""
    def __init__(self, n_classes=7, ndf=64, image_size=64):
        super().__init__()
        self.n_classes = n_classes
        self.image_size = image_size
        self.label_map = nn.Linear(n_classes, image_size * image_size)
        sn = nn.utils.spectral_norm
        self.net = nn.Sequential(
            sn(nn.Conv2d(4, ndf, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)),
        )

    def forward(self, images, labels):
        y = one_hot(labels, self.n_classes).to(images.device)
        label_map = self.label_map(y).view(-1, 1, self.image_size, self.image_size)
        x = torch.cat([images, label_map], dim=1)
        return self.net(x).view(-1, 1)

print("\n✅ Phase 2.3 completed: Conditional Discriminator executed successfully.")


### Phase 3: Medical Classifier

A calibrated ResNet18 provides the rare-class confidence signal used inside the RL reward.

### Phase 3.1: ResNet18 Classifier

The classifier exposes both logits and feature embeddings for evaluation, reward shaping, and t-SNE visualization.

In [ ]:
class ResNet18Classifier(nn.Module):
    """ResNet18 classifier with feature extraction."""
    def __init__(self, n_classes=7, pretrained=True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = models.ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
        try:
            base = models.resnet18(weights=weights)
        except Exception:
            base = models.resnet18(weights=None)
        in_features = base.fc.in_features
        base.fc = nn.Linear(in_features, n_classes)
        self.backbone = base

    def features(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        return self.backbone.fc(self.features(x))

print("\n✅ Phase 3.1 completed: ResNet18 Classifier executed successfully.")


### Phase 3.2: Model Freezing Utility

This helper ensures pretrained networks remain fixed when they are used as evaluators.

In [ ]:
def freeze_model(model):
    """Freeze all model parameters."""
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model

print("\n✅ Phase 3.2 completed: Model Freezing Utility executed successfully.")


### Phase 2.4: Initialize cGAN Networks

Generator and discriminator instances are initialized before adversarial training.

In [ ]:
generator = ConditionalGenerator(LATENT_DIM, N_CLASSES).to(DEVICE)
discriminator = ConditionalDiscriminator(N_CLASSES).to(DEVICE)
generator.apply(weights_init)
discriminator.apply(weights_init)

print("\n⚙️ Model capacity summary: generator and discriminator parameter counts are reported before adversarial training.")
print(sum(p.numel() for p in generator.parameters()), sum(p.numel() for p in discriminator.parameters()))

print("\n✅ Phase 2.4 completed: Initialize cGAN Networks executed successfully.")


### Phase 2.5: R1 Gradient Penalty

R1 regularization is used to improve adversarial training stability.

In [ ]:
def r1_penalty(discriminator, real_images, labels):
    """Compute R1 gradient penalty on real images."""
    # R1 regularization stabilizes the discriminator around real samples.
    real_images = real_images.detach().requires_grad_(True)
    real_logits = discriminator(real_images, labels)
    gradients = torch.autograd.grad(
        outputs=real_logits.sum(),
        inputs=real_images,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    return gradients.pow(2).flatten(1).sum(1).mean()

print("\n✅ Phase 2.5 completed: R1 Gradient Penalty executed successfully.")


### Phase 2.6: cGAN Training Loop

The conditional GAN is trained with balanced batches, label smoothing, mixed precision, and checkpointing.

In [ ]:
def diff_augment(images):
    """Apply lightweight differentiable augmentation before discriminator scoring."""
    if not CFG.get("gan_diff_augment", False):
        return images
    batch = images.size(0)
    brightness = torch.empty(batch, 1, 1, 1, device=images.device).uniform_(-0.12, 0.12)
    contrast = torch.empty(batch, 1, 1, 1, device=images.device).uniform_(0.85, 1.15)
    mean = images.mean(dim=(2, 3), keepdim=True)
    images = (images + brightness - mean) * contrast + mean
    return images.clamp(-1.0, 1.0)


def update_ema_model(ema_model, source_model, decay):
    """Update EMA generator weights."""
    with torch.no_grad():
        for ema_param, source_param in zip(ema_model.parameters(), source_model.parameters()):
            ema_param.data.mul_(decay).add_(source_param.data, alpha=1.0 - decay)
        for ema_buffer, source_buffer in zip(ema_model.buffers(), source_model.buffers()):
            ema_buffer.copy_(source_buffer)


def train_cgan(generator, discriminator, loader, epochs):
    """Train a class-balanced conditional DCGAN."""
    opt_g = torch.optim.Adam(generator.parameters(), lr=CFG.get("gan_lr_g", 1e-4), betas=(0.0, 0.99))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=CFG.get("gan_lr_d", 2e-4), betas=(0.0, 0.99))
    scaler_g = GradScaler(enabled=DEVICE_TYPE == "cuda")
    scaler_d = GradScaler(enabled=DEVICE_TYPE == "cuda")
    ema_generator = copy.deepcopy(generator).to(DEVICE)
    freeze_model(ema_generator)
    ema_decay = CFG.get("gan_ema_decay", 0.999)
    r1_gamma = CFG.get("r1_gamma", 5.0)
    r1_interval = max(1, int(CFG.get("r1_interval", 8)))
    history = []
    best_quality = -float("inf")

    fixed_labels = torch.tensor(
        [CLASS_TO_IDX[name] for name in CLASS_NAMES for _ in range(8)],
        dtype=torch.long,
        device=DEVICE,
    )
    fixed_z = torch.randn(len(fixed_labels), LATENT_DIM, device=DEVICE)

    for epoch in range(1, epochs + 1):
        generator.train()
        discriminator.train()
        epoch_g, epoch_d, epoch_real, epoch_fake = [], [], [], []
        progress = tqdm(loader, desc=f"cGAN epoch {epoch}/{epochs}", leave=False)

        for step, (real_images, labels) in enumerate(progress, 1):
            real_images = real_images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            batch_size = real_images.size(0)

            opt_d.zero_grad(set_to_none=True)
            z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
            with torch.no_grad():
                fake_images = generator(z, labels)

            with autocast(enabled=DEVICE_TYPE == "cuda"):
                real_logits = discriminator(diff_augment(real_images), labels)
                fake_logits = discriminator(diff_augment(fake_images.detach()), labels)
                d_loss = F.relu(1.0 - real_logits).mean() + F.relu(1.0 + fake_logits).mean()

            if step % r1_interval == 0:
                gp = r1_penalty(discriminator, real_images, labels)
                d_loss = d_loss + (0.5 * r1_gamma * r1_interval) * gp

            scaler_d.scale(d_loss).backward()
            scaler_d.unscale_(opt_d)
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=5.0)
            scaler_d.step(opt_d)
            scaler_d.update()

            opt_g.zero_grad(set_to_none=True)
            z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
            with autocast(enabled=DEVICE_TYPE == "cuda"):
                fake_images = generator(z, labels)
                fake_logits_for_g = discriminator(diff_augment(fake_images), labels)
                g_loss = -fake_logits_for_g.mean()

            scaler_g.scale(g_loss).backward()
            scaler_g.unscale_(opt_g)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=5.0)
            scaler_g.step(opt_g)
            scaler_g.update()
            update_ema_model(ema_generator, generator, ema_decay)

            epoch_g.append(float(g_loss.detach().cpu()))
            epoch_d.append(float(d_loss.detach().cpu()))
            epoch_real.append(float(real_logits.detach().mean().cpu()))
            epoch_fake.append(float(fake_logits.detach().mean().cpu()))
            progress.set_postfix(g=np.mean(epoch_g), d=np.mean(epoch_d))

        ema_generator.eval()
        discriminator.eval()
        with torch.no_grad():
            sample_images = ema_generator(fixed_z, fixed_labels)
            sample_logits = discriminator(sample_images, fixed_labels)
            quality_score = float(torch.sigmoid(sample_logits).mean().cpu())

        row = {
            "epoch": epoch,
            "g_loss": np.mean(epoch_g),
            "d_loss": np.mean(epoch_d),
            "real_logit": np.mean(epoch_real),
            "fake_logit": np.mean(epoch_fake),
            "ema_quality_score": quality_score,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(CHECKPOINT_DIR / "gan_history.csv", index=False)

        if quality_score > best_quality:
            best_quality = quality_score
            torch.save(generator.state_dict(), CHECKPOINT_DIR / "generator_best.pt")
            torch.save(ema_generator.state_dict(), CHECKPOINT_DIR / "generator_ema_best.pt")
            torch.save(discriminator.state_dict(), CHECKPOINT_DIR / "discriminator_best.pt")

    torch.save(generator.state_dict(), CHECKPOINT_DIR / "generator_final.pt")
    torch.save(ema_generator.state_dict(), CHECKPOINT_DIR / "generator_ema_final.pt")
    torch.save(discriminator.state_dict(), CHECKPOINT_DIR / "discriminator_final.pt")
    return pd.DataFrame(history)

print("\n✅ Phase 2.6 completed: cGAN Training Loop executed successfully.")

### Phase 2.7: Load, Train, and Freeze cGAN

The best generator and discriminator are loaded or trained, then frozen before reinforcement learning.

In [ ]:
g_path = CHECKPOINT_DIR / "generator_ema_best.pt"
if not g_path.exists():
    g_path = CHECKPOINT_DIR / "generator_best.pt"
d_path = CHECKPOINT_DIR / "discriminator_best.pt"

if USE_CACHED and g_path.exists() and d_path.exists():
    generator.load_state_dict(torch.load(g_path, map_location=DEVICE))
    discriminator.load_state_dict(torch.load(d_path, map_location=DEVICE))
    gan_history = pd.read_csv(CHECKPOINT_DIR / "gan_history.csv") if (CHECKPOINT_DIR / "gan_history.csv").exists() else pd.DataFrame()
else:
    gan_history = train_cgan(generator, discriminator, gan_loader, CFG["gan_epochs"])
    if (CHECKPOINT_DIR / "generator_ema_best.pt").exists():
        generator.load_state_dict(torch.load(CHECKPOINT_DIR / "generator_ema_best.pt", map_location=DEVICE))
    elif (CHECKPOINT_DIR / "generator_ema_final.pt").exists():
        generator.load_state_dict(torch.load(CHECKPOINT_DIR / "generator_ema_final.pt", map_location=DEVICE))

freeze_model(generator)
freeze_model(discriminator)

if len(gan_history):
    print("\n📊 cGAN loss curve: generator and discriminator losses summarize adversarial optimization dynamics across epochs.")
    plt.figure(figsize=(7, 4))
    plt.plot(gan_history["epoch"], gan_history["g_loss"], label="generator")
    plt.plot(gan_history["epoch"], gan_history["d_loss"], label="discriminator")
    plt.title("Figure 2: Conditional GAN Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Hinge adversarial loss")
    plt.legend(title="Network")
    plt.tight_layout()
    plt.show()

    if "ema_quality_score" in gan_history.columns:
        print("\n📊 EMA generator quality curve: discriminator confidence on fixed EMA samples tracks checkpoint quality.")
        plt.figure(figsize=(7, 4))
        plt.plot(gan_history["epoch"], gan_history["ema_quality_score"], label="EMA sample quality")
        plt.title("Figure 2b: EMA Generator Checkpoint Quality")
        plt.xlabel("Epoch")
        plt.ylabel("Mean discriminator realism probability")
        plt.legend(title="Metric")
        plt.tight_layout()
        plt.show()

print("\n✅ Phase 2.7 completed: Load, Train, and Freeze cGAN executed successfully.")

### Phase 2.8: GAN Output Denormalization

Generated tensors are converted from training scale to display scale.

In [ ]:
def denorm_gan_batch(images):
    """Convert GAN tensors from [-1, 1] to [0, 1]."""
    return ((images + 1.0) / 2.0).clamp(0, 1)

print("\n✅ Phase 2.8 completed: GAN Output Denormalization executed successfully.")


### Phase 2.9: Conditional Sample Grid

A visual sanity check compares generated examples across the seven diagnostic labels.

In [ ]:
@torch.no_grad()
def show_gan_samples(generator, n_per_class=4):
    """Show conditional generator samples."""
    generator.eval()
    labels = torch.tensor(
        [CLASS_TO_IDX[name] for name in CLASS_NAMES for _ in range(n_per_class)],
        device=DEVICE,
    )
    z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
    images = denorm_gan_batch(generator(z, labels))
    grid = make_grid(images.cpu(), nrow=n_per_class, padding=2)
    print("\n📊 Conditional sample grid: this visualization qualitatively checks whether the generator responds to diagnostic labels.")
    plt.figure(figsize=(n_per_class * 2, len(CLASS_NAMES) * 2))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title("Figure 3: Conditional cGAN Samples by Diagnostic Class")
    plt.xlabel("Sample column")
    plt.ylabel("Diagnostic class group")
    plt.axis("off")
    plt.show()


show_gan_samples(generator)

print("\n✅ Phase 2.9 completed: Conditional Sample Grid executed successfully.")


### Phase 3.3: Class-Balanced Loss Weights

Inverse-frequency weights reduce the classifier's bias toward majority lesions.

In [ ]:
def class_weights_from_frame(frame, beta=0.999, rare_boost=1.25):
    """Return effective-number class weights for imbalanced medical classification."""
    counts = frame["label"].value_counts().reindex(range(N_CLASSES)).fillna(0).astype(float).values
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-8)
    weights = weights / weights.mean()
    for rare_id in RARE_IDS:
        weights[rare_id] *= rare_boost
    weights[CLASS_TO_IDX["df"]] *= 1.20
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.45, 5.0)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


class RareAwareFocalLoss(nn.Module):
    """Class-balanced focal loss for rare lesion recognition."""
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.view(-1, 1)).squeeze(1)
        pt = log_pt.exp().clamp(min=1e-6, max=1.0)
        loss = -((1.0 - pt) ** self.gamma) * log_pt
        if self.alpha is not None:
            loss = loss * self.alpha[targets]
        return loss.mean()


def set_classifier_backbone_trainable(model, trainable):
    """Freeze or unfreeze ResNet18 feature layers while keeping the classifier head trainable."""
    for name, parameter in model.backbone.named_parameters():
        if name.startswith("fc."):
            parameter.requires_grad_(True)
        else:
            parameter.requires_grad_(trainable)

print("\n✅ Phase 3.3 completed: Class-Balanced Loss Weights executed successfully.")

### Phase 3.4: Classifier Evaluation Function

Validation metrics include both global accuracy and rare-class accuracy.

In [ ]:
@torch.no_grad()
def evaluate_classifier(model, loader, temperature=1.0):
    """Evaluate classifier accuracy and rare-class accuracy."""
    model.eval()
    y_true, y_pred, losses = [], [], []
    criterion = nn.CrossEntropyLoss()

    for images, labels in tqdm(loader, desc="classifier eval", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images) / temperature
        loss = criterion(logits, labels)
        preds = logits.argmax(1)
        losses.append(float(loss.detach().cpu()))
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rare_mask = np.isin(y_true, RARE_IDS)
    per_class_accuracy = {}
    for class_id in range(N_CLASSES):
        class_mask = y_true == class_id
        per_class_accuracy[IDX_TO_CLASS[class_id]] = float((y_true[class_mask] == y_pred[class_mask]).mean()) if class_mask.any() else np.nan
    rare_values = [per_class_accuracy[IDX_TO_CLASS[class_id]] for class_id in RARE_IDS]
    return {
        "loss": float(np.mean(losses)),
        "accuracy": float((y_true == y_pred).mean()),
        "rare_accuracy": float((y_true[rare_mask] == y_pred[rare_mask]).mean()) if rare_mask.any() else np.nan,
        "min_rare_accuracy": float(np.nanmin(rare_values)) if np.isfinite(rare_values).any() else np.nan,
        "df_accuracy": per_class_accuracy.get("df", np.nan),
        "per_class_accuracy": per_class_accuracy,
        "y_true": y_true,
        "y_pred": y_pred,
    }

print("\n✅ Phase 3.4 completed: Classifier Evaluation Function executed successfully.")

### Phase 3.5: Classifier Training Loop

The classifier is fine-tuned with rare-class augmentation, label smoothing, and best-checkpoint selection.

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs):
    """Fine-tune ResNet18 on HAM10000 with rare-aware optimization."""
    criterion = RareAwareFocalLoss(alpha=class_weights_from_frame(train_df), gamma=1.5)
    optimizer = torch.optim.AdamW(
        [
            {"params": [p for name, p in model.named_parameters() if "backbone.fc" not in name], "lr": 5e-5},
            {"params": model.backbone.fc.parameters(), "lr": 3e-4},
        ],
        weight_decay=2e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=max(5, epochs // 3),
        T_mult=1,
        eta_min=8e-7,
    )
    scaler = GradScaler(enabled=DEVICE_TYPE == "cuda")
    history = []
    best_score = -1.0
    warmup_epochs = min(4, max(1, epochs // 6))

    for epoch in range(1, epochs + 1):
        set_classifier_backbone_trainable(model, trainable=epoch > warmup_epochs)
        model.train()
        train_losses = []
        progress = tqdm(train_loader, desc=f"classifier epoch {epoch}/{epochs}", leave=False)

        for images, labels in progress:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=DEVICE_TYPE == "cuda"):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=np.mean(train_losses))

        scheduler.step(epoch)
        metrics = evaluate_classifier(model, val_loader)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": metrics["loss"],
            "val_accuracy": metrics["accuracy"],
            "val_rare_accuracy": metrics["rare_accuracy"],
            "val_min_rare_accuracy": metrics["min_rare_accuracy"],
            "val_df_accuracy": metrics["df_accuracy"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(CHECKPOINT_DIR / "classifier_history.csv", index=False)

        score = (
            0.90 * row["val_accuracy"]
            + 0.85 * row["val_rare_accuracy"]
            + 0.55 * row["val_min_rare_accuracy"]
            + 0.35 * row["val_df_accuracy"]
            - 0.08 * row["val_loss"]
        )
        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), CHECKPOINT_DIR / "classifier_best.pt")

    set_classifier_backbone_trainable(model, trainable=True)
    return pd.DataFrame(history)

print("\n✅ Phase 3.5 completed: Classifier Training Loop executed successfully.")

### Phase 3.6: Temperature Scaling

A scalar temperature calibrates confidence estimates used by the reward function.

In [ ]:
def collect_logits(model, loader):
    """Collect validation logits for temperature scaling."""
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits_list.append(model(images))
            labels_list.append(labels.to(DEVICE, non_blocking=True))
    return torch.cat(logits_list), torch.cat(labels_list)


def fit_temperature(model, loader):
    """Fit one scalar temperature on validation logits."""
    logits, labels = collect_logits(model, loader)
    log_temperature = torch.zeros(1, device=DEVICE, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.05, max_iter=50)

    def closure():
        optimizer.zero_grad(set_to_none=True)
        # Temperature scaling calibrates confidence without retraining the classifier.
        temperature = log_temperature.exp().clamp(0.5, 5.0)
        loss = F.cross_entropy(logits / temperature, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_temperature.exp().clamp(0.5, 5.0).detach().cpu())

print("\n✅ Phase 3.6 completed: Temperature Scaling executed successfully.")


### Phase 3.7: Train, Calibrate, and Validate Classifier

The calibrated classifier is evaluated with a report, confusion matrix, and rare-class accuracy curve.

In [ ]:
classifier = ResNet18Classifier(N_CLASSES, pretrained=True).to(DEVICE)
cls_path = CHECKPOINT_DIR / "classifier_best.pt"

if USE_CACHED and cls_path.exists():
    classifier.load_state_dict(torch.load(cls_path, map_location=DEVICE))
    classifier_history = pd.read_csv(CHECKPOINT_DIR / "classifier_history.csv") if (CHECKPOINT_DIR / "classifier_history.csv").exists() else pd.DataFrame()
else:
    classifier_history = train_classifier(classifier, cls_train_loader, cls_val_loader, CFG["classifier_epochs"])
    if cls_path.exists():
        classifier.load_state_dict(torch.load(cls_path, map_location=DEVICE))

temperature = fit_temperature(classifier, cls_val_loader)
metrics = evaluate_classifier(classifier, cls_val_loader, temperature=temperature)

print("\n📊 Classifier calibration and accuracy: temperature rescales confidence, while validation accuracy measures diagnostic performance.")
print(f"temperature={temperature:.3f}")
print(f"validation_accuracy={metrics['accuracy']:.3f}")
print(f"rare_validation_accuracy={metrics['rare_accuracy']:.3f}")
if metrics["rare_accuracy"] < 0.35:
    print("\n⚙️ Rare-class diagnostic note: rare accuracy remains limited; report this as a dataset-imbalance limitation and rely on PPO-vs-baseline comparison rather than absolute classifier certainty.")
print("\n📊 Classification report: precision, recall, and F1-score summarize class-wise diagnostic separability.")
print(classification_report(metrics["y_true"], metrics["y_pred"], target_names=CLASS_NAMES, digits=3))

print("\n📊 Confusion matrix: rows are true labels and columns are predicted labels, revealing class-level error patterns.")
cm = confusion_matrix(metrics["y_true"], metrics["y_pred"], labels=list(range(N_CLASSES)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
plt.title("Figure 4: Rare-Aware ResNet18 Classifier Confusion Matrix")
plt.xlabel("Predicted diagnostic class")
plt.ylabel("True diagnostic class")
plt.tight_layout()
plt.show()

if len(classifier_history):
    print("\n📊 Classifier accuracy curve: this plot compares global validation accuracy with rare-class validation accuracy over epochs.")
    plt.figure(figsize=(7, 4))
    plt.plot(classifier_history["epoch"], classifier_history["val_accuracy"], label="all classes")
    plt.plot(classifier_history["epoch"], classifier_history["val_rare_accuracy"], label="rare classes")
    plt.title("Figure 5: Rare-Aware Classifier Validation Accuracy by Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Validation accuracy")
    plt.legend(title="Evaluation subset")
    plt.tight_layout()
    plt.show()

freeze_model(classifier)

print("\n✅ Phase 3.7 completed: Train, Calibrate, and Validate Classifier executed successfully.")

### Phase 4: Reinforcement Learning Environment

The RL environment turns latent-space navigation into a continuous-control decision process.

### Phase 4.1: Generated Image Preprocessing

Generated images are resized and normalized so the frozen classifier can evaluate them.

In [ ]:
IMG_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
IMG_STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


def gan_to_classifier(images):
    """Convert generated [-1, 1] images to classifier input."""
    images = denorm_gan_batch(images)
    images = F.interpolate(images, size=(CLS_IMAGE_SIZE, CLS_IMAGE_SIZE), mode="bilinear", align_corners=False)
    return (images - IMG_MEAN) / IMG_STD

print("\n✅ Phase 4.1 completed: Generated Image Preprocessing executed successfully.")


In [ ]:
@torch.no_grad()
def build_real_feature_bank(loader):
    classifier.eval()
    banks = {class_id: [] for class_id in RARE_IDS}
    for images, labels in tqdm(loader, desc="real feature bank", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        features = F.normalize(classifier.features(images), dim=1)
        for class_id in RARE_IDS:
            mask = labels == class_id
            if mask.any():
                banks[class_id].append(features[mask].detach().cpu())

    feature_bank = {}
    prototypes = {}
    for class_id in RARE_IDS:
        if len(banks[class_id]):
            class_features = torch.cat(banks[class_id], dim=0)
        else:
            class_features = torch.zeros(1, 512)
        prototype = F.normalize(class_features.mean(0, keepdim=True), dim=1).squeeze(0)
        feature_bank[class_id] = class_features.to(DEVICE)
        prototypes[class_id] = prototype.to(DEVICE)
    return feature_bank, prototypes


REAL_FEATURE_BANK, REAL_FEATURE_PROTOTYPES = build_real_feature_bank(cls_val_loader)
print("\n✅ Real feature prototypes built for rare-class distribution-aware PPO rewards.")

### Phase 4.2: Custom Gymnasium Environment (`RareGANLatentEnv`)

The agent controls latent-space updates while the frozen GAN, discriminator, and classifier define the state and reward.

In [ ]:
class RareGANLatentEnv(gym.Env):
    """Gymnasium environment for PPO-guided latent-space navigation."""
    metadata = {"render_modes": []}

    def __init__(
        self,
        generator,
        discriminator,
        classifier,
        temperature=1.0,
        rare_ids=None,
        latent_dim=128,
        max_steps=40,
        action_limit=0.18,
        z_clip=3.0,
        reward_threshold=1.00,
    ):
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.classifier = classifier
        self.temperature = max(float(temperature), 0.5)
        self.rare_ids = list(rare_ids if rare_ids is not None else RARE_IDS)
        self.latent_dim = latent_dim
        self.max_steps = max_steps
        self.action_limit = action_limit
        self.z_clip = z_clip
        self.reward_threshold = reward_threshold
        self.rng = np.random.default_rng(SEED)
        self.action_space = spaces.Box(-action_limit, action_limit, shape=(latent_dim,), dtype=np.float32)
        low = np.concatenate([
            np.full(latent_dim, -z_clip),
            np.zeros(N_CLASSES),
            np.array([0.0, 0.0, -1.0, 0.0]),
        ]).astype(np.float32)
        high = np.concatenate([
            np.full(latent_dim, z_clip),
            np.ones(N_CLASSES),
            np.array([1.0, 1.0, 1.0, 1.0]),
        ]).astype(np.float32)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
        self.z = None
        self.target_label = None
        self.steps = 0
        self.prev_confidence = 0.0
        self.last_image = None
        self.last_embedding = None
        self.best_image = None
        self.best_embedding = None
        self.best_reward = -float("inf")
        self.best_info = None

    def _measure(self):
        z_tensor = torch.tensor(self.z[None, :], dtype=torch.float32, device=DEVICE)
        label_tensor = torch.tensor([self.target_label], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            image = self.generator(z_tensor, label_tensor)
            realism = torch.sigmoid(self.discriminator(image, label_tensor)).item()
            classifier_input = gan_to_classifier(image)
            logits = self.classifier(classifier_input) / self.temperature
            probs = F.softmax(logits, dim=1)
            confidence = probs[0, self.target_label].item()
            pred_label = int(probs.argmax(1).item())
            embedding_tensor = F.normalize(self.classifier.features(classifier_input), dim=1)
            prototype = REAL_FEATURE_PROTOTYPES[self.target_label].view(1, -1)
            prototype_similarity = float(torch.sum(embedding_tensor * prototype, dim=1).item())
        self.last_image = image.detach().cpu()[0]
        self.last_embedding = embedding_tensor.detach().cpu()[0]
        return realism, confidence, pred_label, prototype_similarity

    def _observation(self, realism, confidence, prototype_similarity):
        target_one_hot = np.zeros(N_CLASSES, dtype=np.float32)
        target_one_hot[self.target_label] = 1.0
        step_fraction = np.array([self.steps / self.max_steps], dtype=np.float32)
        scores = np.array([realism, confidence, prototype_similarity, step_fraction[0]], dtype=np.float32)
        return np.concatenate([self.z, target_one_hot, scores]).astype(np.float32)

    def _quality_score(self, realism, confidence, pred_label, prototype_similarity, movement_penalty=0.0, boundary_penalty=0.0):
        correct_bonus = 1.0 if pred_label == self.target_label else 0.0
        return (
            0.28 * realism
            + 0.42 * confidence
            + 0.35 * prototype_similarity
            + 0.18 * correct_bonus
            - 0.03 * movement_penalty
            - 0.04 * boundary_penalty
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        options = options or {}
        self.target_label = int(options.get("target_label", self.rng.choice(self.rare_ids)))
        self.z = self.rng.normal(0, 1, size=self.latent_dim).clip(-self.z_clip, self.z_clip).astype(np.float32)
        self.steps = 0
        self.best_reward = -float("inf")
        self.best_image = None
        self.best_embedding = None
        self.best_info = None
        realism, confidence, pred_label, prototype_similarity = self._measure()
        self.prev_confidence = confidence
        quality_score = self._quality_score(realism, confidence, pred_label, prototype_similarity)
        info = {
            "target_label": self.target_label,
            "target_name": IDX_TO_CLASS[self.target_label],
            "realism": realism,
            "confidence": confidence,
            "prototype_similarity": prototype_similarity,
            "pred_label": pred_label,
            "movement_penalty": 0.0,
            "diversity_penalty": 0.0,
            "quality_score": float(quality_score),
            "reward": float(quality_score),
        }
        self.best_reward = float(quality_score)
        self.best_image = self.last_image.clone()
        self.best_embedding = self.last_embedding.clone()
        self.best_info = dict(info)
        return self._observation(realism, confidence, prototype_similarity), info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).clip(-self.action_limit, self.action_limit)
        self.z = np.clip(self.z + action, -self.z_clip, self.z_clip).astype(np.float32)
        self.steps += 1
        realism, confidence, pred_label, prototype_similarity = self._measure()

        confidence_gain = confidence - self.prev_confidence
        movement_penalty = float(np.mean(np.square(action)))
        boundary_penalty = float(np.mean(np.maximum(np.abs(self.z) - 2.4, 0.0) ** 2))
        diversity_penalty = 0.0
        quality_score = self._quality_score(realism, confidence, pred_label, prototype_similarity, movement_penalty, boundary_penalty)
        reward = quality_score + 0.10 * confidence_gain
        self.prev_confidence = confidence
        correct_prediction = pred_label == self.target_label
        terminated = bool((confidence >= 0.78 and correct_prediction and realism >= 0.40 and prototype_similarity >= 0.35) or reward >= self.reward_threshold)
        truncated = bool(self.steps >= self.max_steps)
        info = {
            "target_label": self.target_label,
            "target_name": IDX_TO_CLASS[self.target_label],
            "realism": realism,
            "confidence": confidence,
            "prototype_similarity": prototype_similarity,
            "pred_label": pred_label,
            "movement_penalty": movement_penalty,
            "diversity_penalty": diversity_penalty,
            "quality_score": float(quality_score),
            "reward": float(reward),
        }
        if quality_score > self.best_reward:
            self.best_reward = float(quality_score)
            self.best_image = self.last_image.clone()
            self.best_embedding = self.last_embedding.clone()
            self.best_info = dict(info)
        return self._observation(realism, confidence, prototype_similarity), float(reward), terminated, truncated, info

print("\n✅ Phase 4.2 completed: Custom Gymnasium Environment (RareGANLatentEnv) executed successfully.")

### Phase 4.3: Freeze Verification and API Check

Assertions verify that PPO cannot update evaluator networks, and Gymnasium compliance is checked.

In [ ]:
assert not any(p.requires_grad for p in generator.parameters())
assert not any(p.requires_grad for p in discriminator.parameters())
assert not any(p.requires_grad for p in classifier.parameters())

test_env = RareGANLatentEnv(generator, discriminator, classifier, temperature=temperature)
check_env(test_env, warn=True, skip_render_check=True)
obs, info = test_env.reset(seed=SEED)
print("\n⚙️ Environment validation output: observation shape and reset metadata confirm Gymnasium API compatibility.")
print(obs.shape, info)

print("\n✅ Phase 4.3 completed: Freeze Verification and API Check executed successfully.")


### Phase 5: PPO Training

Stable-Baselines3 PPO learns a policy for reward-guided rare-disease latent navigation.

### Phase 5.1: PPO Reward Logger

The callback tracks mean episode reward and saves the best observed policy checkpoint.

In [ ]:
class RewardLogger(BaseCallback):
    """Log PPO episode reward and save resumable checkpoints."""
    def __init__(self, log_freq=1000, best_path=None, latest_path=None, vecnorm_path=None, history_path=None):
        super().__init__()
        self.log_freq = log_freq
        self.best_path = best_path
        self.latest_path = latest_path
        self.vecnorm_path = vecnorm_path
        self.history_path = history_path
        self.last_log = 0
        self.episode_rewards = []
        self.history = []
        self.best_mean = -float("inf")

    def _save_vecnorm(self):
        vecnorm_env = self.model.get_vec_normalize_env()
        if vecnorm_env is not None and self.vecnorm_path is not None:
            vecnorm_env.save(self.vecnorm_path)

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(float(info["episode"]["r"]))

        if self.num_timesteps - self.last_log >= self.log_freq:
            self.last_log = self.num_timesteps
            recent = self.episode_rewards[-20:]
            mean_reward = float(np.mean(recent)) if recent else np.nan
            self.history.append({"timesteps": self.num_timesteps, "mean_episode_reward": mean_reward})
            print(f"timesteps={self.num_timesteps} mean_episode_reward={mean_reward:.4f}")
            if self.history_path is not None:
                pd.DataFrame(self.history).to_csv(self.history_path, index=False)
            if self.latest_path is not None:
                self.model.save(self.latest_path)
                self._save_vecnorm()
            if recent and mean_reward > self.best_mean:
                self.best_mean = mean_reward
                if self.best_path is not None:
                    self.model.save(self.best_path)
                    self._save_vecnorm()
        return True

print("\n✅ Phase 5.1 completed: PPO Reward Logger executed successfully.")

### Phase 5.2: Vectorized RL Environment

The monitored environment is wrapped with observation and reward normalization for PPO.

In [ ]:
def make_rl_env():
    """Create a monitored RL environment."""
    env = RareGANLatentEnv(generator, discriminator, classifier, temperature=temperature)
    return Monitor(env)


vec_env = DummyVecEnv([make_rl_env])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0, gamma=0.98)

print("\n✅ Phase 5.2 completed: Vectorized RL Environment executed successfully.")

### Phase 5.3: PPO Training and Checkpointing

The PPO agent is loaded from cache or trained with the selected debug/full profile.

In [ ]:
ppo_path = CHECKPOINT_DIR / "ppo_latent_agent.zip"
ppo_latest_path = CHECKPOINT_DIR / "ppo_latest.zip"
ppo_best_path = CHECKPOINT_DIR / "ppo_best.zip"
vecnorm_path = CHECKPOINT_DIR / "ppo_vecnormalize.pkl"
ppo_history_path = CHECKPOINT_DIR / "ppo_reward_history.csv"
callback = RewardLogger(
    log_freq=5000,
    best_path=str(CHECKPOINT_DIR / "ppo_best"),
    latest_path=str(CHECKPOINT_DIR / "ppo_latest"),
    vecnorm_path=str(vecnorm_path),
    history_path=str(ppo_history_path),
)

ppo_history_existing = pd.read_csv(ppo_history_path) if ppo_history_path.exists() else pd.DataFrame()
load_path = None
is_final_model = False

if USE_CACHED and ppo_path.exists() and vecnorm_path.exists():
    load_path = ppo_path
    is_final_model = True
elif USE_CACHED and ppo_latest_path.exists() and vecnorm_path.exists():
    load_path = ppo_latest_path
elif USE_CACHED and ppo_best_path.exists() and vecnorm_path.exists():
    load_path = ppo_best_path

if load_path is not None:
    vec_env = VecNormalize.load(str(vecnorm_path), vec_env)
    model = PPO.load(str(load_path), env=vec_env, device=DEVICE_TYPE)
else:
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=lambda progress_remaining: 8.0e-5 + progress_remaining * 1.7e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.98,
        gae_lambda=0.94,
        clip_range=lambda progress_remaining: 0.12 + progress_remaining * 0.08,
        ent_coef=0.035,
        vf_coef=0.65,
        max_grad_norm=0.5,
        policy_kwargs={"net_arch": {"pi": [512, 256], "vf": [512, 256]}, "activation_fn": nn.Tanh, "ortho_init": True},
        verbose=0,
        seed=SEED,
        device=DEVICE_TYPE,
        tensorboard_log=str(OUTPUT_DIR / "tensorboard"),
    )

remaining_timesteps = max(0, CFG["ppo_timesteps"] - int(getattr(model, "num_timesteps", 0)))
if (not is_final_model) and remaining_timesteps > 0:
    print(f"\n⚙️ PPO training status: running {remaining_timesteps} remaining timesteps toward the configured total of {CFG['ppo_timesteps']}.")
    model.learn(
        total_timesteps=remaining_timesteps,
        callback=callback,
        progress_bar=False,
        reset_num_timesteps=int(getattr(model, "num_timesteps", 0)) == 0,
    )
    model.save(str(ppo_path))
    vec_env.save(str(vecnorm_path))
    ppo_history_new = pd.DataFrame(callback.history)
    ppo_history = pd.concat([ppo_history_existing, ppo_history_new], ignore_index=True)
    if len(ppo_history):
        ppo_history = ppo_history.drop_duplicates(subset=["timesteps"], keep="last")
    ppo_history.to_csv(ppo_history_path, index=False)
else:
    print("\n⚙️ PPO training status: cached PPO checkpoint already satisfies the configured timestep budget.")
    ppo_history = ppo_history_existing

print("\n✅ Phase 5.3 completed: PPO Training and Checkpointing executed successfully.")

### Phase 5.4: PPO Reward Curve

The reward trace summarizes whether latent-space search improves over training.

In [ ]:
vec_env.training = False
vec_env.norm_reward = False

if len(ppo_history):
    print("\n📊 PPO reward curve: mean episode reward indicates whether the policy improves latent-space navigation over training.")
    plt.figure(figsize=(7, 4))
    plt.plot(ppo_history["timesteps"], ppo_history["mean_episode_reward"], label="PPO")
    plt.title("Figure 6: PPO Mean Episode Reward Curve")
    plt.xlabel("Training timesteps")
    plt.ylabel("Mean episode reward")
    plt.legend(title="Policy")
    plt.tight_layout()
    plt.show()

print("\n✅ Phase 5.4 completed: PPO Reward Curve executed successfully.")


### Phase 6: Evaluation and Visualization

Baseline random generation is compared against PPO-guided generation on rare classes.

### Phase 6.1: Evaluation File Utilities

Output folders are reset and real images are standardized for fair comparisons.

In [ ]:
def reset_dir(path):
    """Clear and recreate a directory."""
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_pil_resized(src_path, dst_path, size=GAN_IMAGE_SIZE):
    """Save an RGB image resized for generation evaluation."""
    image = Image.open(src_path).convert("RGB").resize((size, size), Image.BICUBIC)
    image.save(dst_path)

print("\n✅ Phase 6.1 completed: Evaluation File Utilities executed successfully.")


### Phase 6.2: Baseline cGAN Generation

Random latent vectors provide the non-RL baseline for each rare class.

In [ ]:
@torch.no_grad()
def generate_baseline_for_class(class_id, n_images, out_dir, batch_size=64):
    """Generate random-latent cGAN images for one class."""
    out_dir = reset_dir(out_dir)
    label = torch.full((batch_size,), class_id, dtype=torch.long, device=DEVICE)
    produced = 0
    generator.eval()
    while produced < n_images:
        current = min(batch_size, n_images - produced)
        z = torch.randn(current, LATENT_DIM, device=DEVICE)
        labels = label[:current]
        images = denorm_gan_batch(generator(z, labels))
        for idx, image in enumerate(images):
            save_image(image.cpu(), out_dir / f"{produced + idx:04d}.png")
        produced += current
    return out_dir

print("\n✅ Phase 6.2 completed: Baseline cGAN Generation executed successfully.")


### Phase 6.3: PPO-Guided Generation

The trained PPO policy iteratively navigates latent space before each image is saved.

In [ ]:
def normalized_obs_for_policy(obs):
    """Normalize observations with VecNormalize statistics."""
    obs_batch = np.asarray([obs], dtype=np.float32)
    return vec_env.normalize_obs(obs_batch.copy())


def select_diverse_candidates(candidates, n_images, diversity_weight=0.18):
    """Greedily select high-quality candidates while discouraging feature collapse."""
    selected = []
    remaining = list(candidates)
    while remaining and len(selected) < n_images:
        best_idx = 0
        best_score = -float("inf")
        for idx, candidate in enumerate(remaining):
            score = float(candidate["score"])
            if selected:
                similarities = [float(torch.dot(candidate["embedding"], item["embedding"])) for item in selected]
                score -= diversity_weight * max(similarities)
            if score > best_score:
                best_score = score
                best_idx = idx
        selected.append(remaining.pop(best_idx))
    return selected


@torch.no_grad()
def generate_ppo_for_class(class_id, n_images, out_dir):
    """Generate PPO-guided images for one rare class."""
    out_dir = reset_dir(out_dir)
    eval_env = RareGANLatentEnv(generator, discriminator, classifier, temperature=temperature)
    reward_rows = []
    n_restarts = int(CFG.get("ppo_eval_restarts", 1))
    pool_size = int(n_images * CFG.get("ppo_candidate_multiplier", 1))
    candidates = []

    for candidate_idx in tqdm(range(pool_size), desc=f"ppo pool {IDX_TO_CLASS[class_id]}", leave=False):
        selected_image = None
        selected_embedding = None
        selected_info = None
        selected_score = -float("inf")
        selected_episode_reward = 0.0

        for restart_idx in range(n_restarts):
            obs, info = eval_env.reset(options={"target_label": class_id})
            last_info = info
            total_reward = 0.0
            deterministic_policy = restart_idx < max(1, n_restarts // 2)
            for _ in range(eval_env.max_steps):
                action, _ = model.predict(normalized_obs_for_policy(obs), deterministic=deterministic_policy)
                obs, reward, terminated, truncated, last_info = eval_env.step(action[0])
                total_reward += reward
                if terminated or truncated:
                    break

            candidate_image = eval_env.best_image if eval_env.best_image is not None else eval_env.last_image
            candidate_embedding = eval_env.best_embedding if eval_env.best_embedding is not None else eval_env.last_embedding
            candidate_info = eval_env.best_info if eval_env.best_info is not None else last_info
            candidate_score = float(candidate_info.get("quality_score", candidate_info.get("reward", total_reward)))
            if candidate_score > selected_score:
                selected_score = candidate_score
                selected_image = candidate_image.clone()
                selected_embedding = F.normalize(candidate_embedding.float(), dim=0).cpu()
                selected_info = dict(candidate_info)
                selected_episode_reward = total_reward

        candidates.append({
            "image": selected_image,
            "embedding": selected_embedding,
            "info": selected_info,
            "score": selected_score,
            "episode_reward": selected_episode_reward,
        })

    selected_candidates = select_diverse_candidates(candidates, n_images)
    for image_idx, candidate in enumerate(selected_candidates):
        image = denorm_gan_batch(candidate["image"].unsqueeze(0))[0]
        save_image(image, out_dir / f"{image_idx:04d}.png")
        reward_rows.append({"class": IDX_TO_CLASS[class_id], "episode_reward": candidate["episode_reward"], **candidate["info"]})

    return out_dir, pd.DataFrame(reward_rows)

print("\n✅ Phase 6.3 completed: PPO-Guided Generation executed successfully.")

### Phase 6.4: Held-Out Real Rare Images

Validation images are exported class-wise for FID and visual evaluation.

In [ ]:
def prepare_real_dirs():
    """Save held-out real rare images by class."""
    base = reset_dir(GENERATED_DIR / "real")
    dirs = {}
    for class_name in RARE_CLASSES:
        class_dir = reset_dir(base / class_name)
        rows = val_df[val_df["dx"] == class_name].reset_index(drop=True)
        for idx, row in rows.iterrows():
            save_pil_resized(row["path"], class_dir / f"{idx:04d}.png")
        dirs[class_name] = class_dir
    return dirs

print("\n✅ Phase 6.4 completed: Held-Out Real Rare Images executed successfully.")


### Phase 6.5: Generate Evaluation Sets

Real, baseline, and PPO-guided image folders are prepared for downstream metrics.

In [ ]:
n_eval = CFG["eval_per_class"]
real_dirs = prepare_real_dirs()
baseline_dirs = {}
ppo_dirs = {}
ppo_rows = []

for class_name in RARE_CLASSES:
    class_id = CLASS_TO_IDX[class_name]
    baseline_dirs[class_name] = generate_baseline_for_class(class_id, n_eval, GENERATED_DIR / "baseline" / class_name)
    ppo_dirs[class_name], rows = generate_ppo_for_class(class_id, n_eval, GENERATED_DIR / "ppo" / class_name)
    ppo_rows.append(rows)

ppo_generation_log = pd.concat(ppo_rows, ignore_index=True)
ppo_generation_log.to_csv(GENERATED_DIR / "ppo_generation_log.csv", index=False)
print("\n📊 PPO generation diagnostics: class-wise means summarize reward, realism, classifier confidence, prototype similarity, quality score, and diversity penalty after latent navigation.")
diagnostic_columns = [column for column in ["episode_reward", "reward", "quality_score", "realism", "confidence", "prototype_similarity", "diversity_penalty"] if column in ppo_generation_log.columns]
display(ppo_generation_log.groupby("class")[diagnostic_columns].mean())

print("\n✅ Phase 6.5 completed: Generate Evaluation Sets executed successfully.")

### Phase 6.6: Evaluation Image Dataset

Generated folders are wrapped as classifier-ready datasets.

In [ ]:
class EvalImageDataset(Dataset):
    """Image-folder dataset for classifier-based evaluation."""
    def __init__(self, image_dir):
        self.paths = sorted(Path(image_dir).glob("*.png")) + sorted(Path(image_dir).glob("*.jpg"))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        return cls_eval_transform(image)

print("\n✅ Phase 6.6 completed: Evaluation Image Dataset executed successfully.")


### Phase 6.7: Target-Class Recognition Metric

Classifier agreement estimates how often generated images match the requested rare diagnosis.

In [ ]:
@torch.no_grad()
def classifier_target_rate(image_dir, target_label, batch_size=64):
    """Compute target-class rate and mean target confidence."""
    dataset = EvalImageDataset(image_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    preds, confs = [], []
    classifier.eval()
    for images in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = classifier(images) / temperature
        probs = F.softmax(logits, dim=1)
        preds.extend(probs.argmax(1).cpu().numpy().tolist())
        confs.extend(probs[:, target_label].cpu().numpy().tolist())
    preds = np.array(preds)
    return float((preds == target_label).mean()), float(np.mean(confs))

print("\n✅ Phase 6.7 completed: Target-Class Recognition Metric executed successfully.")


### Phase 6.8: Target-Rate Comparison

The target recognition rate compares random cGAN samples against PPO-guided samples.

In [ ]:
target_rows = []
for class_name in RARE_CLASSES:
    class_id = CLASS_TO_IDX[class_name]
    for method, dirs in [("baseline", baseline_dirs), ("ppo", ppo_dirs)]:
        rate, confidence = classifier_target_rate(dirs[class_name], class_id)
        target_rows.append({
            "class": class_name,
            "method": method,
            "target_rate": rate,
            "mean_target_confidence": confidence,
        })

target_rate_table = pd.DataFrame(target_rows)
print("\n📊 Target classification table: higher target rates indicate that generated images are more often recognized as the requested rare disease.")
display(target_rate_table)

print("\n📊 Target-rate comparison plot: this chart contrasts random cGAN sampling with PPO-guided latent search.")
plt.figure(figsize=(7, 4))
sns.barplot(data=target_rate_table, x="class", y="target_rate", hue="method", palette=["#4C78A8", "#F58518"])
plt.title("Figure 7: Target Rare-Disease Classification Rate")
plt.xlabel("Rare diagnostic class")
plt.ylabel("Target classification rate")
plt.legend(title="Generation method")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print("\n✅ Phase 6.8 completed: Target-Rate Comparison executed successfully.")


### Phase 6.9: FID Score Utility

FID quantifies distributional similarity between held-out real rare images and generated images.

In [ ]:
def fid_score(real_dir, fake_dir):
    """Compute FID with pytorch-fid."""
    cmd = [
        sys.executable,
        "-m",
        "pytorch_fid",
        str(real_dir),
        str(fake_dir),
        "--device",
        DEVICE_TYPE,
        "--batch-size",
        "32",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    text = result.stdout + result.stderr
    if "FID:" in text:
        try:
            return float(text.split("FID:")[-1].strip().split()[0])
        except Exception:
            pass
    numbers = []
    for token in text.replace("=", " ").split():
        try:
            numbers.append(float(token))
        except ValueError:
            pass
    return numbers[-1] if numbers else np.nan

print("\n✅ Phase 6.9 completed: FID Score Utility executed successfully.")


### Phase 6.10: FID Directory Preparation

Class-wise folders can be flattened for aggregate rare-class FID computation.

In [ ]:
def make_flat_dir(source_dirs, out_dir):
    """Create a flat image directory from class folders."""
    out_dir = reset_dir(out_dir)
    counter = 0
    for class_name, folder in source_dirs.items():
        for path in sorted(Path(folder).glob("*.png")):
            shutil.copy2(path, out_dir / f"{class_name}_{counter:05d}.png")
            counter += 1
    return out_dir

print("\n✅ Phase 6.10 completed: FID Directory Preparation executed successfully.")


### Phase 6.11: FID Evaluation

FID is computed in full mode and skipped in debug mode to keep iteration fast.

In [ ]:
fid_rows = []
if CFG["run_fid"]:
    for class_name in RARE_CLASSES:
        fid_rows.append({"class": class_name, "method": "baseline", "fid": fid_score(real_dirs[class_name], baseline_dirs[class_name])})
        fid_rows.append({"class": class_name, "method": "ppo", "fid": fid_score(real_dirs[class_name], ppo_dirs[class_name])})

    real_all = make_flat_dir(real_dirs, GENERATED_DIR / "fid_real_all")
    baseline_all = make_flat_dir(baseline_dirs, GENERATED_DIR / "fid_baseline_all")
    ppo_all = make_flat_dir(ppo_dirs, GENERATED_DIR / "fid_ppo_all")
    fid_rows.append({"class": "all_rare", "method": "baseline", "fid": fid_score(real_all, baseline_all)})
    fid_rows.append({"class": "all_rare", "method": "ppo", "fid": fid_score(real_all, ppo_all)})
else:
    print("\n⚙️ FID evaluation skipped: set RUN_MODE=full to compute distributional similarity with pytorch-fid.")

fid_table = pd.DataFrame(fid_rows)
print("\n📊 FID table: lower FID values indicate closer distributional similarity to held-out real rare images.")
display(fid_table)

if len(fid_table):
    print("\n📊 FID comparison plot: this bar chart compares distributional realism between baseline and PPO-guided generation.")
    plt.figure(figsize=(8, 4))
    sns.barplot(data=fid_table, x="class", y="fid", hue="method", palette=["#4C78A8", "#F58518"])
    plt.title("Figure 8: FID Between Held-Out Rare Images and Generated Images")
    plt.xlabel("Diagnostic class")
    plt.ylabel("FID score; lower is better")
    plt.legend(title="Generation method")
    plt.tight_layout()
    plt.show()

print("\n✅ Phase 6.11 completed: FID Evaluation executed successfully.")


### Phase 6.12: Feature Embedding Extraction

Classifier embeddings provide a shared representation for real and generated images.

In [ ]:
@torch.no_grad()
def extract_feature_table(group_dirs, max_per_group=150):
    """Extract ResNet feature embeddings for t-SNE."""
    rows, features = [], []
    classifier.eval()
    for group, dirs in group_dirs.items():
        for class_name, folder in dirs.items():
            paths = sorted(Path(folder).glob("*.png"))[:max_per_group]
            for path in paths:
                image = cls_eval_transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
                feature = F.normalize(classifier.features(image), dim=1).cpu().numpy()[0]
                features.append(feature)
                rows.append({"group": group, "class": class_name, "path": str(path)})
    return np.asarray(features), pd.DataFrame(rows)

print("\n✅ Phase 6.12 completed: Feature Embedding Extraction executed successfully.")


### Phase 6.13: t-SNE Embedding Map

The t-SNE plot visualizes whether PPO-guided samples move closer to real rare images.

In [ ]:
features, feature_frame = extract_feature_table({
    "real": real_dirs,
    "baseline": baseline_dirs,
    "ppo": ppo_dirs,
})

if len(feature_frame) >= 5:
    perplexity = max(2, min(30, (len(feature_frame) - 1) // 3))
    tsne = TSNE(n_components=2, perplexity=perplexity, init="pca", learning_rate="auto", random_state=SEED)
    coords = tsne.fit_transform(features)
    feature_frame["tsne_1"] = coords[:, 0]
    feature_frame["tsne_2"] = coords[:, 1]

    print("\n📊 t-SNE embedding map: this projection compares feature-space proximity among real, baseline, and PPO-guided samples.")
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=feature_frame,
        x="tsne_1",
        y="tsne_2",
        hue="group",
        style="class",
        alpha=0.85,
        s=45,
    )
    plt.title("Figure 9: t-SNE Feature Map of Real and Generated Rare Images")
    plt.xlabel("t-SNE component 1")
    plt.ylabel("t-SNE component 2")
    plt.legend(title="Image group / class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

print("\n✅ Phase 6.13 completed: t-SNE Embedding Map executed successfully.")


### Phase 6.14: Image Grid Utility

A compact 5x5 grid supports qualitative inspection of generated samples.

In [ ]:
def show_image_grid(image_dir, title, n=25):
    """Display a 5x5 image grid from a folder."""
    paths = sorted(Path(image_dir).glob("*.png"))[:n]
    images = [T.ToTensor()(Image.open(path).convert("RGB")) for path in paths]
    if not images:
        return
    grid = make_grid(torch.stack(images), nrow=5, padding=2)
    print(f"\n📊 Image grid: {title}. the grid supports qualitative inspection of generated sample diversity and lesion appearance.")
    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(title)
    plt.xlabel("Sample column")
    plt.ylabel("Sample row")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print("\n✅ Phase 6.14 completed: Image Grid Utility executed successfully.")


### Phase 6.15: Class-Wise Sample Grids

Baseline and PPO-guided images are displayed for every rare disease class.

In [ ]:
for class_name in RARE_CLASSES:
    show_image_grid(baseline_dirs[class_name], f"Baseline cGAN samples: {class_name}")
    show_image_grid(ppo_dirs[class_name], f"PPO-guided samples: {class_name}")

print("\n✅ Phase 6.15 completed: Class-Wise Sample Grids executed successfully.")


### Phase 7: Results Export

The final summary table is saved with checkpoint and generated-image locations.

In [ ]:
summary = target_rate_table.copy()
if len(fid_table):
    summary = summary.merge(fid_table, on=["class", "method"], how="left")
print("\n📊 Final summary table: this dataframe consolidates target-rate and FID metrics for baseline and PPO-guided generation.")
display(summary)

print("\n⚙️ Exporting final artifacts: summary metrics, checkpoints, and generated images are saved for academic review.")
summary.to_csv(OUTPUT_DIR / "rare_generation_summary.csv", index=False)
print(f"checkpoints={CHECKPOINT_DIR}")
print(f"generated_images={GENERATED_DIR}")
print(f"summary={OUTPUT_DIR / 'rare_generation_summary.csv'}")

print("\n✅ Phase 7 completed: Results Export executed successfully.")


## References

- HAM10000: Tschandl, Rosendahl, Kittler, Scientific Data 2018.
- Conditional GAN: Mirza and Osindero, arXiv 1411.1784.
- DCGAN: Radford, Metz, Chintala, arXiv 1511.06434.
- PPO: Schulman et al., arXiv 1707.06347.
- FID: Heusel et al., NeurIPS 2017.